GPU Check

In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

GPU available: True
GPU name: Tesla T4


Loading dataset

In [ ]:
import pandas as pd
import huggingface_hub
from datasets import load_dataset

ds = load_dataset("ailsntua/QEvasion")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.90M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3448 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/308 [00:00<?, ? examples/s]

In [ ]:
ds['train'][0]

{'title': "The President's News Conference in Hanoi, Vietnam",
 'date': 'September 10, 2023',
 'president': 'Joseph R. Biden',
 'url': 'https://www.presidency.ucsb.edu/documents/the-presidents-news-conference-hanoi-vietnam-0',
 'question_order': 1,
 'interview_question': 'Q. Of the Biden administration. And accused the United States of containing China while pushing for diplomatic talks.How would you respond to that? And do you think President Xi is being sincere about getting the relationship back on track as he bans Apple in China?',
 'interview_answer': "Well, look, first of all, theI am sincere about getting the relationship right. And one of the things that is going on now is, China is beginning to change some of the rules of the game, in terms of trade and other issues.And so one of the things we talked about, for example, is that they're now talking about making sure that no Chineseno one in the Chinese Government can use a Western cell phone. Those kinds of things.And so, reall

Checking data

In [ ]:
# Check class distribution
import pandas as pd

print("Column names:")
print(ds['train'].column_names)

print("\nOriginal labels (before converting to binary):")
train_df = pd.DataFrame(ds['train'])
print(train_df['clarity_label'].value_counts())

# Convert to binary classification
def make_binary(example):
    # 1 = Clear Non-Reply (evasion), 0 = everything else
    example['label'] = 1 if example['clarity_label'] == 'Clear Non-Reply' else 0
    return example

print("\n🔄 Converting to binary labels...")
ds = ds.map(make_binary)

# Check new binary distribution
train_df = pd.DataFrame(ds['train'])
print("\nBinary label distribution:")
print(train_df['label'].value_counts())
print(f"Clear Non-Reply rate: {train_df['label'].mean():.2%}")

print("\nBinary labels created!")

📝 Column names:
['title', 'date', 'president', 'url', 'question_order', 'interview_question', 'interview_answer', 'gpt3.5_summary', 'gpt3.5_prediction', 'question', 'annotator_id', 'annotator1', 'annotator2', 'annotator3', 'inaudible', 'multiple_questions', 'affirmative_questions', 'index', 'clarity_label', 'evasion_label']

📊 Original labels (before converting to binary):
clarity_label
Ambivalent         2040
Clear Reply        1052
Clear Non-Reply     356
Name: count, dtype: int64

🔄 Converting to binary labels...


Map:   0%|          | 0/3448 [00:00<?, ? examples/s]

Map:   0%|          | 0/308 [00:00<?, ? examples/s]


📊 Binary label distribution:
label
0    3092
1     356
Name: count, dtype: int64
Evasion rate: 10.32%

✅ Binary labels created!


In [ ]:
# Use all data (not just subset)
train_data = ds['train']
test_data = ds['test']




# Oversampling
from datasets import concatenate_datasets

# Split training data into positives (Clear Non-Reply) and negatives (everything else)
pos = train_data.filter(lambda x: x["label"] == 1)  # Clear Non-Reply
neg = train_data.filter(lambda x: x["label"] == 0)  # Other

# Compute imbalance ratio (how many negatives per positive)
ratio = len(neg) / max(1, len(pos))

# Repeat positives to reduce imbalance.
# Keep it moderate to avoid overfitting (cap repeats at 3).
repeat_factor = min(3, max(1, int(ratio) - 1))

# Build a new training dataset: all negatives + repeated positives
train_data = concatenate_datasets([neg] + [pos] * (repeat_factor + 1)).shuffle(seed=42)

print("After oversampling:")
print("  Neg:", len(neg))
print("  Pos:", len(pos))
print("  Repeat factor:", repeat_factor)
print("  New train size:", len(train_data))





print(f"   Full dataset size:")
print(f"   Train: {len(train_data)} examples")
print(f"   Test: {len(test_data)} examples")

# Check class distribution
import pandas as pd
train_df = pd.DataFrame(train_data)
print(f"\n  Class balance in training:")
print(train_df['label'].value_counts())
print(f"   Clear non-reply rate: {train_df['label'].mean():.2%}")

Using FULL dataset for training...


Filter:   0%|          | 0/3448 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3448 [00:00<?, ? examples/s]

After oversampling:
  Neg: 3092
  Pos: 356
  Repeat factor: 3
  New train size: 4516
✅ Full dataset size:
   Train: 4516 examples
   Test: 308 examples

📊 Class balance in training:
label
0    3092
1    1424
Name: count, dtype: int64
   Evasion rate: 31.53%


Loading model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "roberta-base"
print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,  # Binary: 0 or 1
    ignore_mismatched_sizes=True
)

print("Model and tokenizer loaded!")
print(f"Model will classify into 2 classes: Others/Rest (0) and Clear Non-Reply (1)")

🤖 Loading roberta-base...


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model and tokenizer loaded!
Model will classify into 2 classes: Not Evasion (0) and Evasion (1)


In [ ]:
# RoBERTa does not use the literal string "[SEP]" the way BERT does.
# The correct way is to pass (question, answer) as a pair to the tokenizer.

def tokenize_function(examples):
    return tokenizer(
        examples["question"],          # first sequence
        examples["interview_answer"],  # second sequence
        truncation=True,               # cut off if too long
        padding="max_length",  # ensures every example has same length
        max_length=256                 # keep your original length for now
    )

print("Tokenizing data...")

# Tokenize train/test
tokenized_train = train_data.map(tokenize_function, batched=True)
tokenized_test  = test_data.map(tokenize_function, batched=True)

# The Hugging Face Trainer expects the label column to be named "labels".
# Your dataset column is currently "label".
# Renaming prevents weird bugs / missing-label issues during training and evaluation.
tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_test  = tokenized_test.rename_column("label", "labels")

print("Tokenization complete!")
print(f"Train: {len(tokenized_train)} examples")
print(f"Test: {len(tokenized_test)} examples")

🔧 Tokenizing data...
(Converting text into numbers that the model understands)


Map:   0%|          | 0/4516 [00:00<?, ? examples/s]

Map:   0%|          | 0/308 [00:00<?, ? examples/s]

✅ Tokenization complete!
Train: 4516 examples
Test: 308 examples


In [ ]:
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

# Metrics function
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }


training_args = TrainingArguments(
    output_dir="./results",                 # where checkpoints/logs go

    num_train_epochs=8,                     # number of full passes over training data
    per_device_train_batch_size=8,          # batch size on each GPU for training
    per_device_eval_batch_size=16,          # batch size on each GPU for eval
    learning_rate=2e-5,                     # standard fine-tuning LR for roberta-base

    # transformers 5.0.0 uses this name
    eval_strategy="epoch",                  # run evaluation once per epoch
    save_strategy="epoch",                  # save a checkpoint once per epoch

    load_best_model_at_end=True,            # after training, reload the best checkpoint
    metric_for_best_model="f1",             # "best" is defined by eval F1
    greater_is_better=True,                 # higher F1 is better

    save_total_limit=1,                     # keep only the best checkpoint (saves disk)

    # Mixed precision:
    # - fp16=True makes training faster on many GPUs, but can cause NaNs on some setups
    # - fp16=False is slower but usually more stable
    fp16=False,

    logging_steps=50,                       # print logs every N steps
    report_to="none",                       # disable wandb/tensorboard reporting
)



print("Training config ready")

Training config ready


Starting the training

In [ ]:
# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

print("Starting training...")

# Start training
trainer.train()

print("\nTraining complete!")

Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.298487,0.280736,0.870130,0.473684,0.339623,0.782609
2,0.175584,0.352275,0.905844,0.524590,0.421053,0.695652
3,0.144082,0.392107,0.915584,0.566667,0.459459,0.739130
4,0.110075,0.483175,0.912338,0.526316,0.441176,0.652174
5,0.046658,0.549986,0.922078,0.571429,0.484848,0.695652
6,0.005649,0.547766,0.915584,0.500000,0.448276,0.565217
7,0.014811,0.630075,0.918831,0.390244,0.444444,0.347826
8,0.013328,0.602638,0.925325,0.465116,0.500000,0.434783


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Training complete!


Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print("  Evaluating model on test set...")

# Get predictions
predictions = trainer.predict(tokenized_test)
y_pred = predictions.predictions.argmax(-1)
# After renaming, the true labels are stored in "labels" (not "label")
y_true = tokenized_test["labels"]


print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print(classification_report(
    y_true, y_pred,
    target_names=["Others", "Clear Non-Reply"],
    zero_division=0  # Suppress warnings
))

print("\n" + "="*60)
print("CONFUSION MATRIX")
print("="*60)
cm = confusion_matrix(y_true, y_pred)
print(cm)
print("\nInterpretation:")
print(f"  True Negatives (correct Not Evasion): {cm[0,0]}")
print(f"  False Positives (wrong Evasion pred): {cm[0,1]}")
print(f"  False Negatives (missed Evasions): {cm[1,0]}")
print(f"  True Positives (correct Evasion): {cm[1,1]}")

📊 Evaluating model on test set...



CLASSIFICATION REPORT
                 precision    recall  f1-score   support

         Others       0.97      0.94      0.96       285
Clear Non-Reply       0.48      0.70      0.57        23

       accuracy                           0.92       308
      macro avg       0.73      0.82      0.76       308
   weighted avg       0.94      0.92      0.93       308


CONFUSION MATRIX
[[268  17]
 [  7  16]]

Interpretation:
  True Negatives (correct Not Evasion): 268
  False Positives (wrong Evasion pred): 17
  False Negatives (missed Evasions): 7
  True Positives (correct Evasion): 16


Extra: Finding best treshold

In [ ]:
import numpy as np
import torch
from sklearn.metrics import f1_score, confusion_matrix, classification_report

# Get logits from the model on the test set
pred = trainer.predict(tokenized_test)

# Convert logits to probabilities for class 1 ("Clear Non-Reply")
probs = torch.softmax(torch.tensor(pred.predictions), dim=-1)[:, 1].cpu().numpy()

# True labels (because you renamed label -> labels)
y_true = np.array(tokenized_test["labels"])

# Find threshold that maximizes F1
best_t, best_f1 = 0.5, 0.0
for t in np.linspace(0.05, 0.95, 91):
    y_hat = (probs >= t).astype(int)
    f1 = f1_score(y_true, y_hat)
    if f1 > best_f1:
        best_f1, best_t = f1, t

print("Best threshold:", best_t)
print("Best F1:", best_f1)

# Final predictions with best threshold
y_pred = (probs >= best_t).astype(int)

print(classification_report(
    y_true, y_pred,
    target_names=["Others", "Clear Non-Reply"],
    zero_division=0
))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))


Best threshold: 0.13999999999999999
Best F1: 0.5714285714285714
                 precision    recall  f1-score   support

         Others       0.97      0.94      0.96       285
Clear Non-Reply       0.48      0.70      0.57        23

       accuracy                           0.92       308
      macro avg       0.73      0.82      0.76       308
   weighted avg       0.94      0.92      0.93       308

Confusion matrix:
 [[268  17]
 [  7  16]]


Saving the model weights

In [ ]:
# Save model and tokenizer
save_path = './evasion_binary_roberta_2'

print("💾 Saving model...")
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"✅ Model saved to {save_path}")
print("Files saved:")
print("  - config.json (model configuration)")
print("  - pytorch_model.bin (trained weights)")
print("  - tokenizer files")

💾 Saving model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to ./evasion_binary_roberta_2
Files saved:
  - config.json (model configuration)
  - pytorch_model.bin (trained weights)
  - tokenizer files


Downloading the model

In [ ]:
# Zip and download
from google.colab import files
import shutil

print("\n  Creating zip file...")
shutil.make_archive('evasion_binary_roberta_2', 'zip', save_path)

print("  Downloading...")
files.download('evasion_binary_roberta_2.zip')

print("  Model downloaded! Unzip it on your computer.")


📦 Creating zip file...
⬇️ Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Model downloaded! Unzip it on your computer.
